# Q3: Gaussian Naive Bayes from Scratch - BSDS500 Boundary Detection

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat

BASE = r"C:\Users\cqds\Downloads\bsds500archive"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
IMAGES_TRAIN = os.path.join(BASE, "images", "train")
IMAGES_TEST = os.path.join(BASE, "images", "test")
GT_TRAIN = os.path.join(BASE, "ground_truth", "train")
GT_TEST = os.path.join(BASE, "ground_truth", "test")

### Helper functions for loading images and ground truth

In [ ]:
def list_image_files(images_dir, max_images):
    filenames = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(IMAGE_EXTENSIONS))
    return filenames[:max_images]

def find_gt_path(gt_dir, image_filename):
    stem = os.path.splitext(image_filename)[0]
    candidate = os.path.join(gt_dir, stem + ".mat")
    if not os.path.exists(candidate):
        raise FileNotFoundError(
            f"No matching ground-truth .mat file found for image '{image_filename}' "
            f"(expected {candidate})"
        )
    return candidate

def load_bsds_ground_truth(mat_path):
    mat = loadmat(mat_path)
    gt_struct = mat["groundTruth"]
    n_annotators = gt_struct.shape[1]
    boundary_maps = [np.asarray(gt_struct[0, i]["Boundaries"][0, 0], dtype=float)
                      for i in range(n_annotators)]
    consensus = np.mean(boundary_maps, axis=0)
    return (consensus >= 0.5).astype(int)

def get_boundary_labels(gt_array):
    unique_vals = np.unique(gt_array)
    if len(unique_vals) <= 2:
        return (gt_array > 0).astype(int)
    diff_right = gt_array[:, :-1] != gt_array[:, 1:]
    diff_down = gt_array[:-1, :] != gt_array[1:, :]
    boundary = np.zeros_like(gt_array, dtype=bool)
    boundary[:, :-1] |= diff_right
    boundary[:-1, :] |= diff_down
    return boundary.astype(int)

def extract_pixel_features(img_array):
    gray = img_array.mean(axis=2)
    grad_y = np.abs(np.diff(gray, axis=0, prepend=gray[:1, :]))
    grad_x = np.abs(np.diff(gray, axis=1, prepend=gray[:, :1]))
    grad_mag = np.sqrt(grad_x ** 2 + grad_y ** 2)
    feats = np.stack([img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2], gray, grad_mag], axis=-1)
    return feats.reshape(-1, 5)

def load_pixel_dataset(images_dir, gt_dir, max_images, pixels_per_image, seed=0):
    rng = np.random.RandomState(seed)
    filenames = list_image_files(images_dir, max_images)
    X_parts, y_parts = [], []
    total_pixels = 0
    for fname in filenames:
        img = np.array(Image.open(os.path.join(images_dir, fname)).convert("RGB"), dtype=float) / 255.0
        gt_path = find_gt_path(gt_dir, fname)
        gt = load_bsds_ground_truth(gt_path)
        if gt.shape != img.shape[:2]:
            raise ValueError(
                f"Ground truth shape {gt.shape} does not match image shape "
                f"{img.shape[:2]} for '{fname}'"
            )
        labels_full = get_boundary_labels(gt).reshape(-1)
        feats_full = extract_pixel_features(img)
        total_pixels += labels_full.shape[0]
        boundary_idx = np.where(labels_full == 1)[0]
        nonboundary_idx = np.where(labels_full == 0)[0]
        rng.shuffle(boundary_idx)
        rng.shuffle(nonboundary_idx)
        idx = np.concatenate([boundary_idx[:pixels_per_image], nonboundary_idx[:pixels_per_image]])
        X_parts.append(feats_full[idx])
        y_parts.append(labels_full[idx])
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    return X, y, total_pixels, len(filenames)

feature_names = ["R", "G", "B", "gray", "gradient_magnitude"]

### Load training and test pixel samples

In [ ]:
X_train, y_train, total_train_pixels, n_train_images = load_pixel_dataset(
    IMAGES_TRAIN, GT_TRAIN, max_images=40, pixels_per_image=250, seed=0)
X_test, y_test, total_test_pixels, n_test_images = load_pixel_dataset(
    IMAGES_TEST, GT_TEST, max_images=15, pixels_per_image=150, seed=1)

print("Training images used:", n_train_images, " total pixels:", total_train_pixels)
print("Training sample shape:", X_train.shape, " Test sample shape:", X_test.shape)

### Class priors

In [ ]:
classes = [0, 1]
n_train = len(y_train)
priors = {c: np.mean(y_train == c) for c in classes}
print("P(non-boundary) =", round(priors[0], 4))
print("P(boundary)     =", round(priors[1], 4))

### Per-class, per-feature mean and variance

In [ ]:
means = {}
variances = {}
for c in classes:
    X_c = X_train[y_train == c]
    means[c] = X_c.mean(axis=0)
    variances[c] = X_c.var(axis=0) + 1e-6

stats_table = pd.DataFrame({
    "feature": feature_names,
    "mean_non_boundary": means[0], "var_non_boundary": variances[0],
    "mean_boundary": means[1], "var_boundary": variances[1],
})
print(stats_table)

### Gaussian log-probability density function

In [ ]:
def gaussian_log_pdf(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)

### Prediction function

In [ ]:
def predict(X):
    log_scores = np.zeros((X.shape[0], len(classes)))
    for c in classes:
        log_score = np.log(priors[c]) + np.sum(gaussian_log_pdf(X, means[c], variances[c]), axis=1)
        log_scores[:, c] = log_score
    return np.argmax(log_scores, axis=1), log_scores

### Evaluate on test set

In [ ]:
y_pred, log_scores = predict(X_test)
accuracy = np.mean(y_pred == y_test)
tp = np.sum((y_test == 1) & (y_pred == 1))
tn = np.sum((y_test == 0) & (y_pred == 0))
fp = np.sum((y_test == 0) & (y_pred == 1))
fn = np.sum((y_test == 1) & (y_pred == 0))
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("TP:", tp, "TN:", tn, "FP:", fp, "FN:", fn)
print("Test accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4), " Recall:", round(recall, 4), " F1:", round(f1, 4))

### Plot class-conditional distributions for the gradient-magnitude feature

In [ ]:
grad_idx = feature_names.index("gradient_magnitude")
plt.figure(figsize=(8, 5))
plt.hist(X_train[y_train == 0, grad_idx], bins=40, alpha=0.6, density=True, label="non-boundary")
plt.hist(X_train[y_train == 1, grad_idx], bins=40, alpha=0.6, density=True, label="boundary")
plt.xlabel("gradient magnitude")
plt.ylabel("density")
plt.title("Class-conditional distribution of gradient magnitude")
plt.legend()
plt.grid(alpha=0.3)
plt.show()